# CTI-TAA walkthrough — narrative → threat actor

**Task:** attribute an intrusion narrative to a threat actor.
**Scoring:** synonym-aware **C/P/I** credit (athenabench): 1.0 alias match, 0.5 related group,
0.0 otherwise — using the MISP-galaxy alias + related-group graph. **Cadence:** daily.

In [6]:
import os, sys, json

def _find_root(start):
    d = os.path.abspath(start)
    while d != os.path.dirname(d):
        if os.path.isdir(os.path.join(d, "src", "glokta")):
            return d
        d = os.path.dirname(d)
    raise RuntimeError("could not locate repo root (a dir containing src/glokta)")

ROOT = _find_root(os.getcwd())
sys.path.insert(0, os.path.join(ROOT, "src"))

# Best-effort load of the repo .env so live HF calls have HF_TOKEN; no dotenv dependency.
_envp = os.path.join(ROOT, ".env")
if os.path.exists(_envp):
    for _line in open(_envp):
        _s = _line.strip()
        if _s and not _s.startswith("#") and "=" in _s:
            _k, _v = _s.split("=", 1)
            os.environ.setdefault(_k.strip(), _v.strip())
os.environ.setdefault("TESTING", "1")  # relax settings validators if .env is absent

MODEL = "huggingface/meta-llama/Llama-3.1-8B-Instruct"
LIVE = bool(os.environ.get("HF_TOKEN"))
print("repo root :", ROOT)
print("model     :", MODEL)
print("LIVE calls:", LIVE, "(set HF_TOKEN to enable real inference)")

def run_model(prompt, canned, max_tokens=256):
    """Call the model live if HF_TOKEN is set, else return a canned example response."""
    if LIVE:
        from glokta.infrastructure.cti.inference import complete
        try:
            return complete(MODEL, prompt, max_tokens=max_tokens, timeout=60.0, max_retries=1)
        except Exception as exc:
            print("[live call failed -> canned]", type(exc).__name__, str(exc)[:80])
            return canned
    print("[offline -> canned response]")
    return canned

repo root : /Users/jake/Projects/glokta
model     : huggingface/meta-llama/Llama-3.1-8B-Instruct
LIVE calls: True (set HF_TOKEN to enable real inference)


## 1. Dataflow — advisory → TAA item; galaxy graph resolves synonyms
The label actor is canonicalised; the model's answer is matched against aliases and related groups.

In [7]:
# A parsed CISA advisory (the shape parse_advisory produces). In production this is built from
# the CISA RSS feed + page text; here we use a clean in-memory example.
ADVISORY = {
    "id": "AA24-100A",
    "published": __import__("datetime").date(2024, 4, 9),
    "actor": "APT29",
    "cves": ["CVE-2024-12345"],
    "techniques": ["T1059", "T1566"],
    "text": ("APT29 conducted an espionage campaign attributed with high confidence to the "
             "Russian SVR, exploiting CVE-2024-12345 and using T1059 and T1566."),
    "sections": {
        "Summary": "APT29 attribution and high-confidence assessment (a CONCLUSION).",
        "Technical Details": "The actors used T1059 and T1566; beaconing was observed to 198.51.100.23.",
        "Indicators of Compromise": "198.51.100.23",
        "MITRE ATT&CK Techniques": "T1059, T1566 (a CONCLUSION/mapping)",
        "Mitigations": "Apply vendor patches and enforce MFA.",
    },
}

# Small in-memory reference indices. In production these come from the reference tables via
# build_technique_index() and build_taa_indices(); inline here to keep the notebook DB-free.
TECHNIQUE_INDEX = {"T1059": "T1059", "T1566": "T1566", "T1064": "T1059"}  # T1064 is revoked -> T1059
ALIAS_INDEX = {"apt29": "APT29", "cozy bear": "APT29", "the dukes": "APT29",
               "apt28": "APT28", "fancy bear": "APT28"}
RELATED_INDEX = {"APT29": {"APT28"}, "APT28": {"APT29"}}

In [8]:
from glokta.infrastructure.cti.connectors.report import normalise_report

taa = next(i for i in normalise_report(ADVISORY) if i.task == "taa")
print("input_text:", taa.input_text)
print("label     :", taa.label)        # {'actor': 'APT29'}
print("alias graph (sample):", {k: ALIAS_INDEX[k] for k in list(ALIAS_INDEX)[:3]})
print("related graph       :", RELATED_INDEX)

input_text: APT29 conducted an espionage campaign attributed with high confidence to the Russian SVR, exploiting CVE-2024-12345 and using T1059 and T1566.
label     : {'actor': 'APT29'}
alias graph (sample): {'apt29': 'APT29', 'cozy bear': 'APT29', 'the dukes': 'APT29'}
related graph       : {'APT29': {'APT28'}, 'APT28': {'APT29'}}


## 2. Tasking — prompt + model call

In [9]:
from glokta.infrastructure.cti.prompts import build_prompt, parse_response

prompt = build_prompt("taa", taa.input_text)
print(prompt)
print("-" * 70)
response = run_model(prompt, canned="Answer: Cozy Bear")
print("model response:", repr(response))
print("parsed actor  :", parse_response("taa", response))

You are a threat-intelligence analyst. Based on the intrusion narrative below, name the most likely threat actor responsible.

Narrative:
APT29 conducted an espionage campaign attributed with high confidence to the Russian SVR, exploiting CVE-2024-12345 and using T1059 and T1566.

Respond with only the threat-actor name.
Answer:
----------------------------------------------------------------------
model response: 'APT29'
parsed actor  : APT29


## 3. Scoring — C / P / I credit
An alias of the right actor is **Correct** (1.0); a *related* group is **Plausible** (0.5).

In [10]:
from glokta.infrastructure.cti.evaluator import evaluate_item

ctx = {"alias_index": ALIAS_INDEX, "related_index": RELATED_INDEX}
for name, resp in {
    "alias (C)":     "Answer: Cozy Bear",   # alias of APT29 -> 1.0
    "exact (C)":     "APT29",
    "related (P)":   "APT28",               # related group -> 0.5
    "unknown (I)":   "Answer: Lazarus Group",
}.items():
    s = evaluate_item("taa", taa.label, resp, ctx)
    print(f"{name:12} score={s.score:.2f} result={s.breakdown['result']} pred={s.parsed_output['actor']!r}")

alias (C)    score=1.00 result=C pred='Cozy Bear'
exact (C)    score=1.00 result=C pred='APT29'
related (P)  score=0.50 result=P pred='APT28'
unknown (I)  score=0.00 result=I pred='Lazarus Group'


**Takeaway:** TAA is deliberately not exact-match — naming a *synonym* of the right group is full credit, and naming a *related* group earns partial credit, because real attribution is graph-shaped. (Label quality depends on `detect_actor` precision — a known follow-up.)